# Cancer Mutation Hotspot Analysis

Research-style bioinformatics project using TCGA-like mutation data.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

plt.rcParams['figure.figsize'] = (10,6)


## Load Mutation Data

In [ ]:

# Replace with your file
df = pd.read_csv('mutations.csv')

df.head()


In [ ]:

df = df[['Hugo_Symbol','Protein_Change','Variant_Classification']].dropna()

def get_position(x):
    m = re.search(r'(\d+)', str(x))
    return int(m.group(1)) if m else np.nan

df['Position'] = df['Protein_Change'].apply(get_position)
df.head()


## Mutation Frequency Analysis

In [ ]:

mutation_counts = (
    df.groupby(['Hugo_Symbol','Position'])
      .size()
      .reset_index(name='Frequency')
)

mutation_counts.sort_values('Frequency', ascending=False).head(20)


## Visualization 1: Top Mutation Hotspots

In [ ]:

top20 = mutation_counts.sort_values('Frequency', ascending=False).head(20)

plt.figure(figsize=(12,6))
plt.bar(range(len(top20)), top20['Frequency'])
plt.xticks(range(len(top20)),
           top20['Hugo_Symbol'] + ':' + top20['Position'].astype(str),
           rotation=90)
plt.title('Top 20 Cancer Mutation Hotspots')
plt.ylabel('Mutation Frequency')
plt.show()


## Visualization 2: Most Mutated Genes

In [ ]:

gene_counts = df['Hugo_Symbol'].value_counts().head(15)

gene_counts.plot(kind='bar')
plt.title('Most Frequently Mutated Genes')
plt.ylabel('Mutation Count')
plt.show()


## Visualization 3: Mutation Density of TP53

In [ ]:

tp53 = mutation_counts[mutation_counts['Hugo_Symbol']=='TP53']

plt.hist(tp53['Position'], bins=30)
plt.title('TP53 Mutation Distribution')
plt.xlabel('Protein Position')
plt.ylabel('Frequency')
plt.show()


## Visualization 4: Variant Classification Distribution

In [ ]:

vc = df['Variant_Classification'].value_counts()

plt.pie(vc.values, labels=vc.index, autopct='%1.1f%%')
plt.title('Variant Classification')
plt.show()


## Visualization 5: Heatmap of Gene Hotspots

In [ ]:

top_genes = df['Hugo_Symbol'].value_counts().head(10).index

heat = mutation_counts[mutation_counts['Hugo_Symbol'].isin(top_genes)]

pivot = heat.pivot_table(
    index='Hugo_Symbol',
    columns='Position',
    values='Frequency',
    fill_value=0
)

sns.heatmap(pivot)
plt.title('Cancer Mutation Hotspot Heatmap')
plt.show()


## Hotspot Identification

In [ ]:

hotspots = mutation_counts[mutation_counts['Frequency'] > 100]

hotspots.sort_values('Frequency', ascending=False).head(20)



## Report Questions

1. Which gene contains the highest number of mutations?
2. Which amino acid position is the strongest hotspot?
3. Are mutations clustered in specific protein regions?
4. Which variant classification dominates?
5. Which genes should be prioritized for further study?
